<a href="https://colab.research.google.com/github/shreyaa-008/module-3/blob/main/Module_3_Speak_with_PDF_RAG_Completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 — Speak with the PDF (RAG) Mini App

**Completed OpenRouter version**

Pipeline: **PDF → parsing → chunking → embeddings → vector search → retrieved context → LLM answer**

The notebook also compares a vanilla LLM with the RAG-augmented LLM on three questions.

In [1]:
!pip -q install pypdf openai reportlab numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.5 MB/s eta 0:00:00


In [2]:
import os, re, time, json
import numpy as np
from getpass import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ").strip()
assert OPENROUTER_API_KEY, "Please provide an OpenRouter API key."

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

EMBED_MODEL = "openai/text-embedding-3-small"
CHAT_MODEL = "openai/gpt-4o-mini"

print("API key loaded:", bool(OPENROUTER_API_KEY))
print("Embedding model:", EMBED_MODEL)
print("Chat model:", CHAT_MODEL)

Enter your OpenRouter API key: ··········
API key loaded: True
Embedding model: openai/text-embedding-3-small
Chat model: openai/gpt-4o-mini


## 1. PDF for the playtest

A small sample PDF is generated so the notebook runs immediately. You can replace `PDF_PATH` with a PDF of your choice.

In [3]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

PDF_PATH = "/content/sample_rag_course_notes.pdf"

styles = getSampleStyleSheet()
styles["Title"].alignment = 1
body = styles["BodyText"]
body.leading = 15

paragraphs = [
    ("Speak with PDF: Retrieval-Augmented Generation", styles["Title"]),
    ("Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. A RAG application searches an external knowledge source and supplies relevant passages to a language model before it generates an answer.", body),
    ("A basic RAG pipeline has four stages: parse a document into text, split the text into chunks, convert chunks into numerical embeddings and store them in a vector index, then embed a user's question, retrieve similar chunks, and give those chunks to the language model.", body),
    ("Chunking affects retrieval quality. Very small chunks can lose context, while very large chunks can contain unrelated information and consume more context. Overlap between neighboring chunks helps preserve information near chunk boundaries.", body),
    ("Embeddings represent semantic meaning as vectors. Vector search commonly uses cosine similarity to find chunks whose meaning is close to the user's question, even when the wording differs.", body),
    ("RAG can reduce hallucination when the correct context is retrieved and the prompt tells the model to use only that context. However, if the correct chunk is not retrieved, the model can still fail. RAG also adds parsing, embedding, retrieval, and prompt-construction steps.", body),
    ("RAG systems can be improved with metadata filtering, hybrid search, reranking, caching, query rewriting, and evaluation sets. RAG can also be combined with structured databases or APIs when both unstructured context and precise facts are required.", body),
]

doc = SimpleDocTemplate(PDF_PATH, pagesize=letter, rightMargin=54, leftMargin=54, topMargin=54, bottomMargin=54)
story = []
for text, style in paragraphs:
    story.extend([Paragraph(text, style), Spacer(1, 12)])
doc.build(story)
print("PDF ready:", PDF_PATH)

PDF ready: /content/sample_rag_course_notes.pdf


## 2. Implement `parse_pdf_to_chunks`

In [4]:
from pypdf import PdfReader

def parse_pdf_to_chunks(pdf_path, chunk_size=800, overlap=120):
    if not isinstance(pdf_path, str) or not pdf_path:
        raise ValueError("pdf_path must be a non-empty string")
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    reader = PdfReader(pdf_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append(text)

    full_text = re.sub(r"\s+", " ", " ".join(pages)).strip()
    if not full_text:
        return []

    chunks = []
    step = chunk_size - overlap
    start = 0

    while start < len(full_text):
        end = min(start + chunk_size, len(full_text))
        chunk = full_text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(full_text):
            break
        start += step

    return chunks

chunks = parse_pdf_to_chunks(PDF_PATH, chunk_size=800, overlap=120)
print("Number of chunks:", len(chunks))
for i, c in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ({len(c)} chars) ---\n{c}")

Number of chunks: 3

--- Chunk 0 (800 chars) ---
Speak with PDF: Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) combines information retrieval with text generation. A RAG application searches an external knowledge source and supplies relevant passages to a language model before it generates an answer. A basic RAG pipeline has four stages: parse a document into text, split the text into chunks, convert chunks into numerical embeddings and store them in a vector index, then embed a user's question, retrieve similar chunks, and give those chunks to the language model. Chunking affects retrieval quality. Very small chunks can lose context, while very large chunks can contain unrelated information and consume more context. Overlap between neighboring chunks helps preserve information near chunk boundaries. Embeddings repr

--- Chunk 1 (800 chars) ---
nsume more context. Overlap between neighboring chunks helps preserve information near chunk boundaries. Embeddings repre

## 3. Implement `embed_and_index`

The local NumPy backend is allowed by the assignment, so Pinecone credentials are not required.

In [5]:
def _embed_texts(texts, model):
    if not texts:
        return np.empty((0, 0), dtype=np.float32)

    response = client.embeddings.create(model=model, input=texts)
    ordered = sorted(response.data, key=lambda x: x.index)
    return np.array([x.embedding for x in ordered], dtype=np.float32)

def embed_and_index(chunks, use_pinecone=False, model="text-embedding-3-small"):
    if not chunks:
        raise ValueError("chunks must not be empty")

    model_id = model if "/" in model else "openai/" + model
    vectors = _embed_texts(chunks, model_id)
    ids = [f"doc-{i}" for i in range(len(chunks))]

    return {
        "chunks": list(chunks),
        "ids": ids,
        "vectors": vectors,
        "index_backend": "local",
        "index_ref": "numpy-cosine-index",
        "embed_model": model_id,
        "dim": int(vectors.shape[1]),
        "id_to_chunk": dict(zip(ids, chunks)),
    }

index_obj = embed_and_index(chunks)
print("Backend:", index_obj["index_backend"])
print("Embedding model:", index_obj["embed_model"])
print("Dimensions:", index_obj["dim"])

Backend: local
Embedding model: openai/text-embedding-3-small
Dimensions: 1536


## 4. Vector retrieval

In [6]:
def _cosine_similarity(query_vector, matrix):
    q = np.asarray(query_vector, dtype=np.float32)
    m = np.asarray(matrix, dtype=np.float32)
    qn = np.linalg.norm(q)
    mn = np.linalg.norm(m, axis=1)
    if qn == 0:
        return np.zeros(len(m))
    return (m @ q) / np.maximum(qn * mn, 1e-12)

def retrieve_chunks(question, index_obj, top_k=3):
    q_vector = _embed_texts([question], index_obj["embed_model"])[0]
    scores = _cosine_similarity(q_vector, index_obj["vectors"])
    ranked = sorted(range(len(scores)), key=lambda i: (-float(scores[i]), i))[:top_k]

    return [
        {
            "id": index_obj["ids"][i],
            "chunk": index_obj["chunks"][i],
            "score": float(scores[i]),
            "order": i,
        }
        for i in ranked
    ]

## 5. Implement `answer_query`

In [7]:
def answer_query(question, index_obj, top_k=3, gpt_model="gpt-4o-mini"):
    retrieved = retrieve_chunks(question, index_obj, top_k=top_k)
    context = "\n\n--- CHUNK ---\n\n".join(x["chunk"] for x in retrieved)

    model_id = gpt_model if "/" in gpt_model else "openai/" + gpt_model

    system_prompt = (
        "You are a helpful assistant answering questions about a PDF. "
        "Use ONLY the provided context. Do not use outside knowledge. "
        "If the answer is not supported by the context, respond exactly: "
        "I don't know based on the document."
    )

    user_prompt = f"Question:\n{question}\n\nContext:\n{context}"

    response = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()

print(answer_query("What are the four main stages of the RAG pipeline?", index_obj))

The four main stages of the RAG pipeline are: parse a document into text, split the text into chunks, convert chunks into numerical embeddings and store them in a vector index, then embed a user's question, retrieve similar chunks, and give those chunks to the language model.


# 6. Three-question playtest: Vanilla LLM vs RAG

In [8]:
QUESTIONS = [
    "What are the four main stages of the RAG pipeline?",
    "Why does chunking use overlap, and what can happen with chunks that are too small or too large?",
    "What limitations and improvement techniques for RAG are described in the document?",
]

def vanilla_answer(question):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "Answer the question using general knowledge. You do not have access to the PDF."},
            {"role": "user", "content": question},
        ],
        temperature=0.0,
        max_tokens=350,
    )
    return response.choices[0].message.content.strip()

results = []

for i, question in enumerate(QUESTIONS, 1):
    print("=" * 80)
    print(f"QUESTION {i}: {question}")

    t0 = time.perf_counter()
    vanilla = vanilla_answer(question)
    vanilla_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    rag = answer_query(question, index_obj, top_k=3)
    rag_time = time.perf_counter() - t1

    print("\nVANILLA LLM:")
    print(vanilla)
    print(f"Time: {vanilla_time:.2f}s")

    print("\nRAG:")
    print(rag)
    print(f"Time: {rag_time:.2f}s")

    print("\nRETRIEVED CHUNKS:")
    for item in retrieve_chunks(question, index_obj, 3):
        print(f"- {item['id']} | similarity={item['score']:.4f}")

    results.append({
        "question": question,
        "vanilla": vanilla,
        "rag": rag,
        "vanilla_time": vanilla_time,
        "rag_time": rag_time,
    })

QUESTION 1: What are the four main stages of the RAG pipeline?

VANILLA LLM:
The RAG (Retrieval-Augmented Generation) pipeline typically consists of four main stages:

1. **Retrieval**: In this stage, relevant documents or pieces of information are retrieved from a large corpus or database based on a user's query. This is often done using techniques such as keyword matching, semantic search, or other information retrieval methods.

2. **Ranking**: After retrieval, the documents are ranked based on their relevance to the query. This can involve scoring the documents using various algorithms to determine which ones are most likely to provide useful information for the next stage.

3. **Generation**: In this stage, a generative model (often a language model) takes the ranked documents and the original query to produce a coherent and contextually relevant response. The model synthesizes information from the retrieved documents to generate a final output.

4. **Output**: The final stage inv

# 7. Reflection — 500–700 words

This activity demonstrated how Retrieval-Augmented Generation can change the behavior of a language model by supplying external context at query time. I compared a vanilla language model response with a RAG response for three questions about the same PDF. The vanilla model answered from its general learned knowledge, while the RAG system first retrieved relevant document chunks and then passed those chunks to the model. The RAG responses were more closely tied to the wording and facts of the PDF, while the vanilla responses could contain general explanations that were not necessarily present in the document.

The biggest benefit of RAG was grounding. For questions about the four stages of the RAG pipeline, chunking, embeddings, and system limitations, the retrieved passages gave the model specific information to use. This reduces the opportunity for unsupported details because the system prompt explicitly tells the model to answer only from the supplied context. RAG does not completely eliminate hallucination, however. If the correct information is not retrieved, the model may still fail. Therefore, retrieval quality is just as important as generation quality.

Chunking played an important role in retrieval. The PDF was normalized and divided into overlapping chunks of approximately 800 characters with 120 characters of overlap. Very small chunks can separate related ideas and remove useful context. Very large chunks can mix unrelated information and use more of the model's context window. The overlap helps preserve information that appears near the boundary between two chunks. This experiment showed that chunk size is not simply a preprocessing setting; it directly affects what the vector search system can retrieve.

Embeddings provided the semantic search layer. Each document chunk was converted into a numerical vector using the OpenRouter embeddings endpoint, and the user's question was embedded with the same model. Cosine similarity was then used to rank the chunks. This allows retrieval to match meaning rather than requiring exact wording. The retrieved chunks were inserted into the generation prompt, creating the complete RAG flow from document to answer.

There are trade-offs. RAG requires extra steps compared with a normal LLM call: PDF parsing, chunking, embedding, vector storage, retrieval, and prompt construction. These steps can increase latency and implementation complexity. Embedding and generation requests can also add cost. On the other hand, the extra work provides a way to answer questions about private or changing information without retraining the model. A local NumPy index was sufficient for this mini app, while a production system could use Pinecone or another vector database.

I would extend the application in several ways. First, I would add caching so repeated document embeddings and repeated questions do not require unnecessary API calls. Second, I would add hybrid retrieval that combines vector similarity with keyword search. Third, I would use reranking to improve the ordering of retrieved chunks before sending them to the LLM. Finally, a RAG+KAG design could combine unstructured PDF retrieval with structured knowledge from a database or API. This would be useful when an application needs both document context and precise structured facts.

Overall, the activity showed that RAG is not simply adding a PDF to an LLM. It is a pipeline in which parsing, chunking, embeddings, retrieval, prompt construction, and generation all influence the final answer. Good retrieval can make answers more relevant and grounded, but poor retrieval can still produce weak results. This makes evaluation and careful system design essential.